In [ ]:
%load_ext autoreload
%autoreload 2
import sys
import os
ProjDIR = "/home/jw3514/Work/CellType_Psy/CellTypeBias_VIP/" # Change to your project directory
sys.path.insert(1, f'{ProjDIR}/src/')
sys.path.insert(1, '/home/jw3514/Work/UNIMED/src')
from CellType_PSY import *
from UNIMED import *
#import scanpy as sc
HGNC, ENSID2Entrez, GeneSymbol2Entrez, Entrez2Symbol = LoadGeneINFO()

try:
    os.chdir(f"{ProjDIR}/notebooks/")
    print(f"Current working directory: {os.getcwd()}")
except FileNotFoundError as e:
    print(f"Error: Could not change directory - {e}")
except Exception as e:
    print(f"Unexpected error: {e}")

In [ ]:
HumanCT_OverallEXP = pd.read_csv("/home/jw3514/Work/CellType_Psy/dat2/ExpMatch/HumanCT.MatchDF.csv", index_col=0)

In [ ]:
HumanCT_OverallEXP.head(200)

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(np.log10(HumanCT_OverallEXP["Exp"]+1), bins=100, color='cornflowerblue', alpha=0.7, edgecolor='black')
plt.xlabel('Log10(Expression + 1)', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.title('Distribution of Gene Expression Values', fontsize=14, pad=15)
plt.grid(True, alpha=0.3)
plt.tight_layout()

In [ ]:
HumanCT_OverallEXP["Exp"].sample(n=10, random_state=42)

In [ ]:
LowExpCut = 10000
LowExpCutLog = np.log10(LowExpCut+1)
print(LowExpCutLog)
LowExpGenes = HumanCT_OverallEXP[HumanCT_OverallEXP["Exp"] < LowExpCut].index.tolist()
NGenesLowExp = len(LowExpGenes)
print(f"Number of genes with expression < {LowExpCut}: {NGenesLowExp}")

In [ ]:
common_genes = list(set(HumanCT_OverallEXP.index.values) & set(HumanCT_Spec.index))
GeneExp = np.log10(HumanCT_OverallEXP.loc[common_genes, "Exp"]+1)
GeneAvgSpec = HumanCT_Spec.loc[common_genes, :].mean(axis=1)
print(len(GeneExp), len(GeneAvgSpec))

In [ ]:
EFFECT_Col = "EFFECT"
EFFECT_Adj_Col = "EFFECT_Adj"
values1 = Anno.sort_index()["Total UMI"].values
values2 = HIQ_Z2_Bias.sort_index()[EFFECT_Col].values
plot_correlation(values2, values1, "UMI", "HIQ ASD HCT", dpi=80)

values1 = Anno.sort_index()["Total UMI"].values
values2 = SCZ_Z2_Bias.sort_index()[EFFECT_Col].values
plot_correlation(values2, values1, "UMI", "SCZ HCT", dpi=80)

values1 = Anno.sort_index()["Total UMI"].values
values2 = LIQ_Z2_Bias.sort_index()[EFFECT_Col].values
plot_correlation(values2, values1, "UMI", "LIQ ASD HCT", dpi=80)

values1 = Anno.sort_index()["Total UMI"].values
values2 = DDD_hc_Bias_top61.sort_index()[EFFECT_Col].values
plot_correlation(values2, values1, "UMI", "DDD HCT top61", dpi=80)


In [ ]:
fig = plt.figure(dpi=200)
plt.scatter(GeneExp, GeneAvgSpec, s=0.1)
plt.xlabel("Average Expression")
plt.ylabel("Average Specificity")
plt.title("Average Expression vs Average Specificity")
#plt.ylim(0.0015, 0.0022)
plt.show()

In [ ]:
HumanCT_Spec = pd.read_csv("/home/jw3514/Work/CellType_Psy/dat/Test.BiasMat/HumanCT.Spec.clip.csv", index_col=0)
HumanCT_Spec.columns = HumanCT_Spec.columns.astype(int)
HumanCT_Spec = HumanCT_Spec.loc[~HumanCT_Spec.index.isin(LowExpGenes)]
HumanCT_Spec.to_csv("/home/jw3514/Work/CellType_Psy/dat/Test.BiasMat/HumanCT.Spec.clip.noLowExp.cut1e4.csv")

In [ ]:
GeneWeightDIR = "../dat/GeneWeights/"
HIQ_GW = Fil2Dict("{}/HIQ.top61.nopLI.LGD_Dmis_SameWeight.gw".format(GeneWeightDIR))
SCZ_GW = Fil2Dict("{}/SCZ.top61.nopLI.LGD_Dmis_SameWeight.exclude_Mis2.gw".format(GeneWeightDIR))

In [ ]:
HIQ_ASD_Genes = list(HIQ_GW.keys())
SCZ_Genes = list(SCZ_GW.keys())

In [ ]:
print("number of low exp genes in HIQ GW: {}".format(len([g for g in HIQ_ASD_Genes if g in LowExpGenes])))
print("number of low exp genes in SCZ GW: {}".format(len([g for g in SCZ_Genes if g in LowExpGenes])))

In [ ]:
SCZ_lowexp = [g for g in SCZ_Genes if g in LowExpGenes]
SCZ_lowexp

In [ ]:
HumanCT_Spec = pd.read_csv("/home/jw3514/Work/CellType_Psy/dat/Test.BiasMat/HumanCT.Spec.clip.csv", index_col=0)
HumanCT_Z2 = pd.read_csv("/home/jw3514/Work/CellType_Psy/dat/HumanCTExpressionMats/Human.Cluster.Log2Mean.Z1clip5.Z2.clip3.Dec30.csv", index_col=0)
HumanCT_Spec.columns = HumanCT_Spec.columns.astype(int)
HumanCT_Z2.columns = HumanCT_Z2.columns.astype(int)

In [ ]:
CGE_idx = Anno[Anno["Supercluster"] == "CGE interneuron"].index

In [ ]:
SCZ_Genes_Common = [g for g in SCZ_Genes if g in HumanCT_Spec.index]
SCZ_Genes_Common = [g for g in SCZ_Genes if g in HumanCT_Z2.index]

In [ ]:
CGE_idx = Anno[Anno["Supercluster"] == "CGE interneuron"].index
SCZ_Gene_Z2 = HumanCT_Z2.loc[SCZ_Genes, CGE_idx].mean(axis=1)
SCZ_Gene_Spec = HumanCT_Spec.loc[SCZ_Genes, CGE_idx].mean(axis=1)

plt.figure(figsize=(8,6))

# Plot points in different colors based on whether gene is in SCZ_lowexp
for gene, z2, spec in zip(SCZ_Genes, SCZ_Gene_Z2, SCZ_Gene_Spec):
    if gene in SCZ_lowexp:
        plt.scatter(z2, spec, color='red', alpha=0.6)
    else:
        plt.scatter(z2, spec, color='blue', alpha=0.6)

# Add horizontal line at y=1/461
plt.axhline(y=1/461, color='black', linestyle='--', alpha=0.5)

# Add vertical line at x=0 
plt.axvline(x=0, color='black', linestyle='--', alpha=0.5)

plt.xlabel('Expression Z-score')
plt.ylabel('Specificity Score')
plt.title('SCZ Gene Expression vs Specificity in CGE Interneurons')

In [ ]:
Common_Genes = [x for x in HumanCT_Z2.index.values if x in HumanCT_Spec.index.values]

In [ ]:
def plot_gene_expression_vs_specificity(CT_Index, common_genes, gene_z2, gene_spec, geneset):
    """
    Plot gene expression vs specificity scores with marginal distributions.
    
    Args:
        CT_Index: List of cell type indices
        common_genes: List of gene IDs
        gene_z2: DataFrame with z-score expression values
        gene_spec: DataFrame with specificity scores  
        geneset: List of genes to highlight
    """
    # Calculate mean scores across CGE interneurons
    all_gene_z2 = gene_z2.loc[common_genes, CT_Index].mean(axis=1)
    all_gene_spec = gene_spec.loc[common_genes, CT_Index].mean(axis=1)

    # Create figure with gridspec
    fig = plt.figure(figsize=(8,8))
    gs = fig.add_gridspec(3, 3)

    # Create main scatter plot
    ax_scatter = fig.add_subplot(gs[1:, :-1])

    # Create boolean mask for genes in geneset
    is_highlight = [gene in geneset for gene in common_genes]

    # Plot non-highlighted genes
    ax_scatter.scatter(all_gene_z2[~np.array(is_highlight)], all_gene_spec[~np.array(is_highlight)], 
               color='grey', alpha=0.6, label='Non-SCZ genes')

    # Plot highlighted genes
    ax_scatter.scatter(all_gene_z2[np.array(is_highlight)], all_gene_spec[np.array(is_highlight)],
               color='red', alpha=0.6, label='SCZ genes')

    # Add horizontal line at mean specificity
    ax_scatter.axhline(y=np.mean(all_gene_spec), color='black', linestyle='--', alpha=0.5)

    # Add vertical line at x=0 
    ax_scatter.axvline(x=0, color='black', linestyle='--', alpha=0.5)

    ax_scatter.set_xlabel('Expression Z-score')
    ax_scatter.set_ylabel('Specificity Score')
    ax_scatter.set_xlim(-5, 5)

    # Create distribution plots
    ax_histx = fig.add_subplot(gs[0, :-1])
    ax_histy = fig.add_subplot(gs[1:, -1])

    # Plot X-axis distribution
    ax_histx.hist(all_gene_z2[~np.array(is_highlight)], bins=50, alpha=0.5, color='grey', density=True)
    ax_histx.hist(all_gene_z2[np.array(is_highlight)], bins=50, alpha=0.5, color='red', density=True)
    ax_histx.set_xlim(ax_scatter.get_xlim())
    ax_histx.set_xticks([])

    # Plot Y-axis distribution
    ax_histy.hist(all_gene_spec[~np.array(is_highlight)], bins=50, orientation='horizontal', alpha=0.5, color='grey', density=True)
    ax_histy.hist(all_gene_spec[np.array(is_highlight)], bins=50, orientation='horizontal', alpha=0.5, color='red', density=True)
    ax_histy.set_ylim(ax_scatter.get_ylim())
    ax_histy.set_yticks([])

    plt.suptitle('SCZ Gene Expression vs Specificity in CGE Interneurons')
    plt.tight_layout()
    
    return fig


In [ ]:
# Call the function with the data
CGE_idx = Anno[Anno["Supercluster"] == "CGE interneuron"].index
fig = plot_gene_expression_vs_specificity(CGE_idx, Common_Genes, HumanCT_Z2, HumanCT_Spec, SCZ_Genes)

In [ ]:
# Call the function with the data
#Supercluster = "CGE interneuron"
Supercluster = "Medium spiny neuron"
CGE_idx = Anno[Anno["Supercluster"] == Supercluster].index
fig = plot_gene_expression_vs_specificity(CGE_idx, Common_Genes, HumanCT_Z2, HumanCT_Spec, SCZ_Genes)

In [ ]:
# Call the function with the data
#Supercluster = "CGE interneuron"
Supercluster = "Astrocyte"
CGE_idx = Anno[Anno["Supercluster"] == Supercluster].index
fig = plot_gene_expression_vs_specificity(CGE_idx, Common_Genes, HumanCT_Z2, HumanCT_Spec, SCZ_Genes)

In [ ]:
# Call the function with the data
#Supercluster = "CGE interneuron"
Supercluster = "Upper rhombic lip"
CGE_idx = Anno[Anno["Supercluster"] == Supercluster].index
fig = plot_gene_expression_vs_specificity(CGE_idx, Common_Genes, HumanCT_Z2, HumanCT_Spec, SCZ_Genes)

In [ ]:
# Call the function with the data
#Supercluster = "CGE interneuron"
Supercluster = "Lower rhombic lip"
CGE_idx = Anno[Anno["Supercluster"] == Supercluster].index
fig = plot_gene_expression_vs_specificity(CGE_idx, Common_Genes, HumanCT_Z2, HumanCT_Spec, SCZ_Genes)

In [ ]:
### Z-score demon

In [ ]:
def generate_random_sum_one(fixed_val=0.5):
    # Initialize array with 0.5 as first element
    result = [fixed_val]
    
    # Generate 9 random numbers for remaining positions
    remaining = 1 - fixed_val  # Since 0.5 is fixed
    for i in range(8):  # Generate 8 numbers, leaving space for last adjustment
        if remaining <= 0:
            val = 0
        else:
            val = np.random.uniform(0, remaining)
            remaining -= val
        result.append(val)
    
    # Add final number to make sum exactly 1
    result.append(remaining)
    
    # Shuffle the array (keeping one 0.5)
    non_fixed = result[1:]
    np.random.shuffle(non_fixed)
    result[1:] = non_fixed
    result.sort(reverse=True)
    return result


In [ ]:
for i in range(3):
    random_numbers = generate_random_sum_one(0.5)
    zscore = (random_numbers - np.mean(random_numbers)) / np.std(random_numbers)
    print(zscore)


In [ ]:
for i in range(3):
    random_numbers = generate_random_sum_one(0)
    zscore = (random_numbers - np.mean(random_numbers)) / np.std(random_numbers)
    print(zscore)

In [ ]:
## Adjust HumanCT_Spec = pd.read_csv("/home/jw3514/Work/CellType_Psy/dat/Test.BiasMat/HumanCT.Spec.clip.csv", index_col=0)

In [ ]:
HumanCT_Spec_adj = HumanCT_Spec.copy(deep=True)

In [ ]:
# Normalize each row to sum to 1
HumanCT_Spec_adj = HumanCT_Spec_adj.div(HumanCT_Spec_adj.sum(axis=1), axis=0)
#HumanCT_Spec_adj

In [ ]:
ExpL = pd.read_csv("/home/jw3514/Work/CellType_Psy/dat/HumanCTExpressionMats/Human.Cluster.UMI.Exp.withFilt.Apr18.csv", index_col=0)
common_indices = list(set(ExpL.index.values) & set(HumanCT_Spec_adj.index))
ExpL = ExpL.loc[common_indices,:]

In [ ]:
GeneAvgExp = np.log10(ExpL.mean(axis=1)+1)
Test2 = HumanCT_Spec_adj.loc[common_indices,:]
GeneAvgSpec = Test2.mean(axis=1)

In [ ]:
plt.scatter(GeneAvgExp, GeneAvgSpec, s=0.1)
plt.xlabel("Average Expression")
plt.ylabel("Average Specificity")
plt.title("Average Expression vs Average Specificity")
#plt.ylim(0.0015, 0.0022)
plt.show()

In [ ]:
HumanCT_Spec_adj.mean(axis=1)

In [ ]:
HumanCT_Spec_adj.mean(axis=0)

In [ ]:
HumanCT_Spec_adj.to_csv("/home/jw3514/Work/CellType_Psy/dat/Test.BiasMat/HumanCT.Spec.clip.adj.csv")


In [ ]:
# Call the function with the data
CGE_idx = Anno[Anno["Supercluster"] == "CGE interneuron"].index
fig = plot_gene_expression_vs_specificity(CGE_idx, Common_Genes, HumanCT_Z2, HumanCT_Spec_adj, SCZ_Genes)